<a href="https://colab.research.google.com/github/swiz9/Flight-Delay-Prediction-System/blob/ann-model/FNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount Google Drive to access files stored in your Drive from Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from google.colab import drive
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
# Mount the Google Drive folder to access files from Drive in Colab
drive.mount('/content/drive')
# Load the flight dataset from Google Drive and display the first five rows
path = "/content/drive/MyDrive/flights_sample_3m.zip"
df = pd.read_csv(path)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,0.0,186.0,176.0,153.0,1065.0,NaN,NaN,NaN,NaN,NaN
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,0.0,235.0,236.0,189.0,1399.0,NaN,NaN,NaN,NaN,NaN
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,0.0,118.0,112.0,87.0,680.0,NaN,NaN,NaN,NaN,NaN
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",...,0.0,260.0,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",...,0.0,181.0,182.0,153.0,985.0,NaN,NaN,NaN,NaN,NaN


In [4]:
# Remove irrelevant or unnecessary columns from the dataset to simplify analysis
def drop_irrelevant_columns(data):
    columns_to_drop = ['CANCELLED', 'CANCELLATION_CODE', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON',
                       'TAXI_IN', 'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS',
                       'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT', 'DOT_CODE', 'AIRLINE_CODE',
                       'ORIGIN_CITY', 'DEST_CITY', 'AIRLINE_DOT', 'FL_NUMBER', 'DIVERTED']
    return data.drop(columns=[col for col in columns_to_drop if col in data.columns], axis=1)

# Extract useful time-based features (day, month, hour, etc.) from the flight date column
def extract_date_features(data):
    if 'FL_DATE' in data.columns:
        data['FL_DATE'] = pd.to_datetime(data['FL_DATE'], errors='coerce')
        data['DayOfWeek'] = data['FL_DATE'].dt.dayofweek
        data['Month'] = data['FL_DATE'].dt.month
        data['Day'] = data['FL_DATE'].dt.day
        data['Hour'] = data['CRS_DEP_TIME'].astype(str).str.zfill(4).str[:2].astype(int)
        data.drop(columns=['FL_DATE'], inplace=True)  # Drop original date column
    return data

In [5]:
# Function to encode categorical columns
def encode_categorical_columns(data):
    le = LabelEncoder()
    list_of_labels = ['AIRLINE', 'ORIGIN', 'DEST']

    for label in list_of_labels:
        if label in data.columns:
            data[label] = le.fit_transform(data[label])
    return data
# Function to handle missing values
def handle_missing_values(data):
    imputer = SimpleImputer(strategy='mean')
    return pd.DataFrame(imputer.fit_transform(data), columns=data.columns)

# Preprocessing pipeline
def preprocess_data(data):
    data = drop_irrelevant_columns(data)
    data = extract_date_features(data)
    data = encode_categorical_columns(data)
    data = handle_missing_values(data)
    return data

# Preprocess data
df = preprocess_data(df)

In [6]:
# Set features and target
features = df.drop(columns=['DEP_DELAY'])
target = np.where(df['DEP_DELAY'] > 15, 1, 0)
# Split data into train, val, and test sets
X_train, X_temp, y_train, y_temp = train_test_split(features, target, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


In [7]:
# Scale the features
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)

# Scale the target (delay times)
scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1))
y_val_scaled = scaler_y.transform(y_val.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1))


In [8]:
# Defineclassification model
model = Sequential()

# Input layer
model.add(Dense(128, input_dim=X_train_scaled.shape[1], activation='relu'))

# Hidden layers with dropout
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))


# Output layer for
model.add(Dense(1))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001),
              loss='mean_squared_error',
              metrics=['mae'])

In [10]:
# Early stopping to avoid overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [17]:
# Train the model
history = model.fit(X_train_scaled, y_train_scaled,
                    validation_data=(X_val_scaled, y_val_scaled),
                    epochs=50, batch_size=128)


Epoch 1/50
14063/14063 ━━━━━━━━━━━━━━━━━━━━ 44s 3ms/step - loss: 0.0102 - mae: 0.0503 - val_loss: 0.0043 - val_mae: 0.0166
Epoch 2/50
14063/14063 ━━━━━━━━━━━━━━━━━━━━ 46s 3ms/step - loss: 0.0104 - mae: 0.0508 - val_loss: 0.0035 - val_mae: 0.0197
Epoch 3/50
14063/14063 ━━━━━━━━━━━━━━━━━━━━ 46s 3ms/step - loss: 0.0101 - mae: 0.0503 - val_loss: 0.0030 - val_mae: 0.0163
Epoch 4/50
14063/14063 ━━━━━━━━━━━━━━━━━━━━ 79s 3ms/step - loss: 0.0102 - mae: 0.0507 - val_loss: 0.0041 - val_mae: 0.0137
Epoch 5/50
14063/14063 ━━━━━━━━━━━━━━━━━━━━ 43s 3ms/step - loss: 0.0098 - mae: 0.0498 - val_loss: 0.0066 - val_mae: 0.0158
Epoch 6/50
14063/14063 ━━━━━━━━━━━━━━━━━━━━ 45s 3ms/step - loss: 0.0102 - mae: 0.0505 - val_loss: 0.0059 - val_mae: 0.0179
Epoch 7/50
14063/14063 ━━━━━━━━━━━━━━━━━━━━ 44s 3ms/step - loss: 0.0100 - mae: 0.0502 - val_loss: 0.0031 - val_mae: 0.0189
Epoch 8/50
14063/14063 ━━━━━━━━━━━━━━━━━━━━ 44s 3ms/step - loss: 0.0100 - mae: 0.0501 - val_loss: 0.0058 - val_mae: 0.0176
Epoch 9/50
14063